In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings("ignore")
import re
import unicodedata
from scipy import stats
import geopandas as gpd

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option("display.max_seq_items", None)

BASE_DIR      = r"C:\Users\Usuario\OneDrive - Global Green Growth Institute\Documentos\2025\Outputs\Output1\Stress Test\3.Data"
MENSUALES_DIR = os.path.join(BASE_DIR, "Escenarios Cambio Climatico IDEAM IV comunicacion", "Mensuales")
WEB_DATA_DIR  = os.path.join(BASE_DIR, "Scripts Python", "webpage_climate", "data")

os.chdir(BASE_DIR)
print("Directorio de trabajo :", os.getcwd())
print("Carpeta web/data      :", WEB_DATA_DIR)

In [ ]:
pip install geopandas

In [ ]:
pip install sodapy

## Precipitación histórica – Carga y procesamiento

In [ ]:
precipitacion = pd.read_csv("Precipitación_20251222.csv")

for col in ["Latitud", "Longitud", "ValorObservado"]:
    precipitacion[col] = (
        precipitacion[col]
        .astype(str)
        .str.replace(",", ".", regex=False)
        .str.strip()
        .replace("nan", None)
        .astype(float)
    )

fecha_str_ = precipitacion["FechaObservacion"].str.slice(0, 11)
precipitacion["fecha"] = pd.to_datetime(
    fecha_str_, format="%Y %b %d", errors="coerce", cache=True
)

In [ ]:
station_cols = [
    "CodigoEstacion", "NombreEstacion", "Departamento",
    "Municipio", "ZonaHidrografica", "Latitud", "Longitud"
]

df_daily = (
    precipitacion
    .groupby(station_cols + ["fecha"], as_index=False)
    .agg(
        precip_min_10min=("ValorObservado", "min"),
        precip_max_10min=("ValorObservado", "max"),
        precip_media_10min=("ValorObservado", "mean"),
        precip_acum_diaria=("ValorObservado", "sum")
    )
)

df_daily = df_daily.sort_values(["CodigoEstacion", "fecha"])
print(df_daily.shape)
df_daily.head()

In [ ]:
data_2 = pd.read_csv("Precipitación_20251222_2.csv", decimal=",")

for col in ["Latitud", "Longitud", "ValorObservado"]:
    data_2[col] = data_2[col].astype(float)

fecha_str = data_2["FechaObservacion"].str.slice(0, 11)
data_2["fecha"] = pd.to_datetime(
    fecha_str, format="%Y %b %d", errors="coerce", cache=True
)

In [ ]:
df_daily_2 = (
    data_2
    .groupby(station_cols + ["fecha"], as_index=False)
    .agg(
        precip_min_10min=("ValorObservado", "min"),
        precip_max_10min=("ValorObservado", "max"),
        precip_media_10min=("ValorObservado", "mean"),
        precip_acum_diaria=("ValorObservado", "sum")
    )
)

df_daily_2 = df_daily_2.sort_values(["CodigoEstacion", "fecha"])
print(df_daily_2.shape)
df_daily_2.head()

In [ ]:
df_daily_vf = pd.concat([df_daily, df_daily_2])
df_daily_vf = df_daily_vf.drop_duplicates(["CodigoEstacion", "fecha"])
print("df_daily_vf shape:", df_daily_vf.shape)
df_daily_vf.head()

In [ ]:
df_daily_vf['fecha'] = pd.to_datetime(df_daily_vf['fecha'])
df_daily_vf['mes']   = df_daily_vf['fecha'].dt.month

# ── 1. Percentil 95 por estación y mes (umbral histórico) ─────────────────
p95 = (
    df_daily_vf
    .groupby(['CodigoEstacion', 'mes'])['precip_acum_diaria']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'precip_acum_diaria': 'p95'})
)

df_daily_vf = df_daily_vf.merge(p95, on=['CodigoEstacion', 'mes'], how='left')
df_daily_vf['extremo'] = (df_daily_vf['precip_acum_diaria'] > df_daily_vf['p95']).astype(int)

# ── 2. Frecuencia histórica de extremos por estación ─────────────────────
station_cols_alerta = ['CodigoEstacion', 'NombreEstacion', 'Departamento', 'Municipio', 'Latitud', 'Longitud']

estaciones_alerta = (
    df_daily_vf
    .groupby(station_cols_alerta)['extremo']
    .mean()
    .reset_index()
    .rename(columns={'extremo': 'frecuencia_extremos'})
)

# ── 3. Tendencia en los últimos 2 años ────────────────────────────────────
fecha_max   = df_daily_vf['fecha'].max()
fecha_corte = fecha_max - pd.DateOffset(years=2)
df_reciente = df_daily_vf[df_daily_vf['fecha'] >= fecha_corte].copy()

df_reciente['anio_mes'] = df_reciente['fecha'].dt.to_period('M').dt.to_timestamp()

extremos_mensuales = (
    df_reciente
    .groupby(['CodigoEstacion', 'anio_mes'])['extremo']
    .agg(n_extremos='sum', n_dias='count')
    .reset_index()
)
extremos_mensuales['tasa_extremos'] = extremos_mensuales['n_extremos'] / extremos_mensuales['n_dias']

def calcular_tendencia(grupo):
    """Regresión lineal de la tasa mensual de extremos en el tiempo."""
    if len(grupo) < 4:
        return pd.Series({'pendiente': 0.0, 'p_valor': 1.0, 'tendencia': 'insuficiente'})
    x = np.arange(len(grupo))
    y = grupo['tasa_extremos'].values
    slope, _, _, p_value, _ = stats.linregress(x, y)
    if p_value < 0.05:
        direccion = 'creciente' if slope > 0 else 'decreciente'
    else:
        direccion = 'estable'
    return pd.Series({'pendiente': slope, 'p_valor': p_value, 'tendencia': direccion})

tendencias = (
    extremos_mensuales
    .groupby('CodigoEstacion')
    .apply(calcular_tendencia)
    .reset_index()
)

# ── 4. Ratio reciente vs histórico ────────────────────────────────────────
freq_reciente = (
    extremos_mensuales
    .groupby('CodigoEstacion')['tasa_extremos']
    .mean()
    .reset_index()
    .rename(columns={'tasa_extremos': 'frecuencia_reciente'})
)

estaciones_alerta = (
    estaciones_alerta
    .merge(tendencias[['CodigoEstacion', 'pendiente', 'p_valor', 'tendencia']], on='CodigoEstacion', how='left')
    .merge(freq_reciente, on='CodigoEstacion', how='left')
)

estaciones_alerta['ratio_reciente'] = (
    estaciones_alerta['frecuencia_reciente']
    / estaciones_alerta['frecuencia_extremos'].replace(0, np.nan)
)

# ── 5. Alerta compuesta (4 niveles) ───────────────────────────────────────
UMBRAL_FREQ   = 0.10   # >10 % de días históricos sobre p95
UMBRAL_RATIO  = 1.20   # últimos 2 años tienen ≥20 % más extremos que el histórico

def categorizar_alerta(row):
    alta_freq       = row['frecuencia_extremos'] > UMBRAL_FREQ
    creciente       = row['tendencia'] == 'creciente'
    aceleracion     = pd.notna(row['ratio_reciente']) and row['ratio_reciente'] > UMBRAL_RATIO
    if alta_freq and (creciente or aceleracion):
        return 'CRÍTICA'
    elif alta_freq:
        return 'ALTA'
    elif creciente or aceleracion:
        return 'MODERADA'
    else:
        return 'BAJA'

NIVEL_NUMERICO = {'BAJA': 0, 'MODERADA': 1, 'ALTA': 2, 'CRÍTICA': 3}

estaciones_alerta['alerta_compuesta'] = estaciones_alerta.apply(categorizar_alerta, axis=1)
estaciones_alerta['alerta']           = estaciones_alerta['alerta_compuesta'].map(NIVEL_NUMERICO)

# ── 6. Resumen ────────────────────────────────────────────────────────────
print("=== Distribución de alerta compuesta ===")
print(estaciones_alerta['alerta_compuesta'].value_counts())
print("\n=== Tendencias últimos 2 años ===")
print(estaciones_alerta['tendencia'].value_counts())

cols_resumen = [
    'CodigoEstacion', 'NombreEstacion', 'Departamento', 'Municipio',
    'frecuencia_extremos', 'frecuencia_reciente', 'ratio_reciente',
    'tendencia', 'p_valor', 'alerta_compuesta', 'alerta'
]
estaciones_alerta[cols_resumen].sort_values('alerta', ascending=False).head(20)

In [ ]:
# Carga davipola y proyecta a EPSG 3116
davipola = pd.read_excel(os.path.join(MENSUALES_DIR, "davipola_dane.xlsx"))

gdf_mun = gpd.GeoDataFrame(
    davipola,
    geometry=gpd.points_from_xy(davipola.LONGITUD, davipola.LATITUD),
    crs="EPSG:4326",
).to_crs(epsg=3116)

# Convierte estaciones_alerta a GeoDataFrame y proyecta
gdf_estaciones = gpd.GeoDataFrame(
    estaciones_alerta,
    geometry=gpd.points_from_xy(estaciones_alerta.Longitud, estaciones_alerta.Latitud),
    crs="EPSG:4326"
).to_crs(epsg=3116)

# Asigna el municipio más cercano a cada estación
gdf_est_muni = gpd.sjoin_nearest(
    gdf_mun[['COD_DPTO', 'NOM_DPTO', 'COD_MPIO', 'NOM_MPIO', 'geometry']],
    gdf_estaciones,
    how='right',
    distance_col="dist_m"
)

# ── Helpers robustos ante NaN ──────────────────────────────────────────────
def modo_seguro(serie, default='BAJA'):
    """Moda ignorando NaN; si todo es NaN retorna default."""
    vc = serie.dropna().value_counts()
    return vc.idxmax() if not vc.empty else default

def tendencia_muni(serie):
    """Prioriza 'creciente'; si no, toma la moda; si vacío, 'sin_datos'."""
    vals = serie.dropna()
    if vals.empty:
        return 'sin_datos'
    if 'creciente' in vals.values:
        return 'creciente'
    vc = vals.value_counts()
    return vc.idxmax() if not vc.empty else 'sin_datos'

# Agrega alertas a nivel municipal
df_alerta_municipal = (
    gdf_est_muni
    .groupby(['COD_DPTO', 'NOM_DPTO', 'COD_MPIO', 'NOM_MPIO'], as_index=False)
    .agg(
        alerta=('alerta', 'max'),
        alerta_compuesta=('alerta_compuesta', modo_seguro),
        frecuencia_extremos=('frecuencia_extremos', 'max'),
        frecuencia_reciente=('frecuencia_reciente', 'max'),
        ratio_reciente=('ratio_reciente', 'max'),
        tendencia=('tendencia', tendencia_muni),
        n_estaciones=('CodigoEstacion', 'count'),
    )
)

print(f"Municipios con cobertura de estaciones: {len(df_alerta_municipal):,}")
print("\n=== Alerta histórica municipal ===")
print(df_alerta_municipal['alerta_compuesta'].value_counts())

# ── Guardar salidas ───────────────────────────────────────────────────────
# 1. Nivel estación → consumido por datos precipitacion diarios.ipynb
estaciones_alerta['cod_norm'] = estaciones_alerta['CodigoEstacion'].astype(str).str.strip().str.lstrip('0')
out_estaciones = os.path.join(WEB_DATA_DIR, "alerta_historica_estaciones.csv")
estaciones_alerta.to_csv(out_estaciones, index=False, encoding="utf-8-sig")
print("\nGuardado (estaciones):", out_estaciones)

# 2. Nivel municipal → salida principal de este notebook
out_municipal = os.path.join(WEB_DATA_DIR, "alerta_historica_municipal.csv")
df_alerta_municipal.to_csv(out_municipal, index=False, encoding="utf-8-sig")
print("Guardado (municipal) :", out_municipal)

df_alerta_municipal.sort_values('alerta', ascending=False).head(20)

## Escenarios mensuales de precipitación 2021-2100 (SSP126)

In [ ]:
precipitacion_2021_2100_m = pd.read_csv(
    os.path.join(MENSUALES_DIR, "DatosMensuales_Precipitacion_SSP126_2021-2100.txt"),
    sep="\t", dtype=str, engine="python", header=None, decimal='.'
)

fixed_cols = [
    'ID', 'Longitud', 'Latitud', 'Departamentos', 'Municipios',
    'Areas Hidrograficas', 'Zonas Hidrograficas', 'SubZonas Hidrograficas',
    'RegionPluvHomog_v1_26Reg', 'RegionPluvHomog_v2_14Reg'
]

n_total_cols = precipitacion_2021_2100_m.shape[1]
n_date_cols = n_total_cols - len(fixed_cols)

dates = pd.date_range(start="2021-01-01", periods=n_date_cols, freq="MS")
date_cols = dates.strftime("%Y/%m").tolist()

data_mensual = precipitacion_2021_2100_m
data_mensual.columns = fixed_cols + date_cols

def clean_text(x):
    if not isinstance(x, str):
        return x
    try:
        x = x.encode("latin1").decode("utf-8")
    except:
        pass
    x = re.sub(r"[\x00-\x1f\x7f-\x9f]", "", x)
    x = unicodedata.normalize("NFC", x)
    return x.strip()

text_cols = data_mensual.select_dtypes(include="object").columns
data_mensual[text_cols] = data_mensual[text_cols].applymap(clean_text)

gdf_mensual_precipitacion = gpd.GeoDataFrame(
    data_mensual,
    geometry=gpd.points_from_xy(data_mensual.Longitud, data_mensual.Latitud),
    crs="EPSG:4326"
)

print("gdf_mensual_precipitacion shape:", gdf_mensual_precipitacion.shape)
gdf_mensual_precipitacion.head()

In [ ]:
gdf_mensual_precipitacion_3116 = gdf_mensual_precipitacion.to_crs(epsg=3116)

gdf_municipios = gpd.sjoin_nearest(
    gdf_mun,
    gdf_mensual_precipitacion_3116,
    distance_col="dist_m"
)

print("gdf_municipios shape:", gdf_municipios.shape)
gdf_municipios.head()

In [ ]:
out_path = os.path.join(WEB_DATA_DIR, "escenarios_mensuales.csv")
gdf_municipios.drop(columns="geometry").to_csv(out_path, index=False, encoding="utf-8-sig")
print("Guardado:", out_path)